# Legacy Migration Analysis Cookbook

> Practical Jupyter notebook for users migrating from openstudio-server
> to OSimFlow. Shows how to load, query, and visualise campaign results
> using pandas and DuckDB — replacing MongoDB queries and PAT GUI charts.

## What this notebook covers

1. Loading `aggregated_results.parquet` (and CSV) with pandas and DuckDB
2. Replicating common MongoDB queries from openstudio-server
3. Generating parallel coordinate plots and scatter plots
4. Boilerplate snippets you can copy into your own analysis

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

# Point this at your campaign output directory
RESULTS_DIR = Path("results")

# Prefer Parquet (faster, typed), fall back to CSV
parquet_path = RESULTS_DIR / "aggregated_results.parquet"
csv_path = RESULTS_DIR / "aggregated_results.csv"

if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
    print(f"Loaded {len(df)} rows from Parquet")
elif csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} rows from CSV")
else:
    print("No results found. Run a campaign first with `osimflow run ...`")
    print("This notebook will use synthetic demo data for illustration.")
    # Generate synthetic demo data
    np.random.seed(42)
    n = 200
    df = pd.DataFrame(
        {
            "sample_id": [f"{i:04d}" for i in range(n)],
            "status": ["completed"] * n,
            "insul_r": np.random.uniform(5, 30, n),
            "wwr_south": np.random.uniform(0.2, 0.8, n),
            "cooling_setpoint": np.random.uniform(22, 28, n),
            "lighting_power": np.random.uniform(8, 15, n),
            "hvac_type": np.random.choice(["VAV", "FCU", "GSHP", "PTAC"], n),
        }
    )
    # Create correlated EUI
    df["eui_kwh_m2_yr"] = (
        300
        - 3.5 * df["insul_r"]
        + 80 * df["wwr_south"]
        + 2.5 * df["cooling_setpoint"]
        + 4.0 * df["lighting_power"]
        + np.random.normal(0, 15, n)
    )
    df.loc[df["sample_id"].isin(["0005", "0042", "0099"]), "status"] = "failed"
    print(f"Using {len(df)} synthetic demo rows")

df.head(10)

---
## 1. Basic Exploration

In openstudio-server, you would start by browsing the web dashboard.
Here, we use pandas to get the same overview.

In [ ]:
# --- Overview (replaces: MongoDB db.data_points.count()) ---
print(f"Total samples: {len(df)}")
print(f"Completed: {(df['status'] == 'completed').sum()}")
print(f"Failed: {(df['status'] == 'failed').sum()}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nBasic statistics:\n{df.describe()}")

In [ ]:
# --- Failed simulations (replaces: db.data_points.find({"status.state": "failed"})) ---
failed = df[df["status"] == "failed"]
print(f"Failed samples ({len(failed)}):")
if len(failed) > 0:
    # In real output, check results/failed_simulations.csv for error messages
    print(failed[["sample_id", "status"]])

---
## 2. Replicating MongoDB Queries

Each cell shows the MongoDB query pattern from openstudio-server
and its pandas equivalent.

In [ ]:
# --- MongoDB: All completed data points with EUI ---
# db.data_points.find(
#   {"status.state": "completed"},
#   {"name": 1, "results.eui": 1}
# )

completed = df[df["status"] == "completed"]
completed[["sample_id", "eui_kwh_m2_yr"]].head(20)

In [ ]:
# --- MongoDB: Average EUI by parameter bin ---
# db.data_points.aggregate([
#   { $match: { "status.state": "completed" } },
#   { $bucket: {
#       groupBy: "$measure_attributes.insul_r",
#       boundaries: [5, 10, 15, 20, 25, 30],
#       output: { avgEUI: { $avg: "$results.eui" } }
#   }}
# ])

completed = df[df["status"] == "completed"].copy()
completed["insul_r_bin"] = pd.cut(completed["insul_r"], bins=[5, 10, 15, 20, 25, 30])
bin_summary = completed.groupby("insul_r_bin", observed=True)["eui_kwh_m2_yr"].agg(
    ["mean", "count"]
)
bin_summary.columns = ["avg_eui", "n_samples"]
bin_summary

In [ ]:
# --- MongoDB: Top 10 samples by EUI ---
# db.data_points.find(
#   {"status.state": "completed"},
#   {"name": 1, "results.eui": 1}
# ).sort({"results.eui": -1}).limit(10)

completed.nlargest(10, "eui_kwh_m2_yr")[["sample_id", "eui_kwh_m2_yr"]]

In [ ]:
# --- MongoDB: Distinct values of a categorical variable ---
# db.data_points.distinct("measure_attributes.hvac_type")

df["hvac_type"].unique()

In [ ]:
# --- MongoDB: Count by HVAC type ---
# db.data_points.aggregate([
#   { $group: { _id: "$measure_attributes.hvac_type", count: { $sum: 1 } } }
# ])

df["hvac_type"].value_counts()

---
## 3. DuckDB for Large Campaigns

For campaigns with >10,000 samples, DuckDB provides fast SQL queries
directly on Parquet files without loading everything into memory.

In [ ]:
import duckdb

con = duckdb.connect()

# If you have a real Parquet file, query it directly:
# result = con.execute("""
#     SELECT * FROM read_parquet('results/aggregated_results.parquet')
#     WHERE status = 'completed'
# """).fetchdf()

# For demo, query the in-memory DataFrame
result = con.execute("""
    SELECT
        hvac_type,
        COUNT(*) as n_samples,
        ROUND(AVG(eui_kwh_m2_yr), 1) as mean_eui,
        ROUND(MIN(eui_kwh_m2_yr), 1) as min_eui,
        ROUND(MAX(eui_kwh_m2_yr), 1) as max_eui,
        ROUND(STDDEV(eui_kwh_m2_yr), 1) as sd_eui
    FROM df
    WHERE status = 'completed'
    GROUP BY hvac_type
    ORDER BY mean_eui
""").fetchdf()

result

In [ ]:
# DuckDB: Parameter sensitivity — correlation with EUI
sensitivity = con.execute("""
    SELECT
        'insul_r' as parameter,
        ROUND(CORR(insul_r, eui_kwh_m2_yr), 3) as pearson_r
    FROM df WHERE status = 'completed'
    UNION ALL
    SELECT
        'wwr_south',
        ROUND(CORR(wwr_south, eui_kwh_m2_yr), 3)
    FROM df WHERE status = 'completed'
    UNION ALL
    SELECT
        'cooling_setpoint',
        ROUND(CORR(cooling_setpoint, eui_kwh_m2_yr), 3)
    FROM df WHERE status = 'completed'
    UNION ALL
    SELECT
        'lighting_power',
        ROUND(CORR(lighting_power, eui_kwh_m2_yr), 3)
    FROM df WHERE status = 'completed'
    ORDER BY ABS(pearson_r) DESC
""").fetchdf()

sensitivity

---
## 4. Visualisations

These plots replace the PAT GUI's built-in charts. Copy any cell
as a starting point for your own analysis.

In [ ]:
# --- EUI Histogram (replaces PAT: Results > Histogram) ---
fig, ax = plt.subplots(figsize=(10, 6))
completed = df[df["status"] == "completed"]

ax.hist(completed["eui_kwh_m2_yr"], bins=30, color="steelblue", edgecolor="white")
ax.axvline(
    completed["eui_kwh_m2_yr"].mean(),
    color="red",
    linestyle="--",
    label=f"Mean: {completed['eui_kwh_m2_yr'].mean():.1f} kWh/m²/yr",
)
ax.set_xlabel("EUI (kWh/m²/yr)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Distribution of EUI Across All Samples", fontsize=14)
ax.legend(fontsize=11)
fig.tight_layout()
plt.show()

In [ ]:
# --- EUI vs Parameter Scatter (replaces PAT: Results > Scatter Plot) ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
param_cols = ["insul_r", "wwr_south", "cooling_setpoint", "lighting_power"]
param_labels = [
    "Insulation R-value",
    "Window-to-Wall Ratio (South)",
    "Cooling Setpoint (°C)",
    "Lighting Power (W/m²)",
]

for ax, col, label in zip(axes.flat, param_cols, param_labels, strict=False):
    ax.scatter(completed[col], completed["eui_kwh_m2_yr"], alpha=0.5, s=20, color="steelblue")
    # Add trend line
    z = np.polyfit(completed[col], completed["eui_kwh_m2_yr"], 1)
    p = np.poly1d(z)
    x_sorted = np.sort(completed[col])
    ax.plot(x_sorted, p(x_sorted), color="red", linewidth=2)
    # Correlation
    r = completed[col].corr(completed["eui_kwh_m2_yr"])
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel("EUI (kWh/m²/yr)", fontsize=10)
    ax.set_title(f"r = {r:.3f}", fontsize=11)

fig.suptitle("EUI vs Design Parameters", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# --- Parallel Coordinates Plot (replaces PAT: Results > Parallel Coordinates) ---
from pandas.plotting import parallel_coordinates

# Select numeric parameter columns
param_cols = ["insul_r", "wwr_south", "cooling_setpoint", "lighting_power"]
kpi_col = "eui_kwh_m2_yr"

# Normalise all columns to [0, 1] for visual comparison
norm_df = completed[param_cols + [kpi_col]].copy()
for col in norm_df.columns:
    col_min, col_max = norm_df[col].min(), norm_df[col].max()
    if col_max > col_min:
        norm_df[col] = (norm_df[col] - col_min) / (col_max - col_min)

# Bin EUI into categories for colour-coding
norm_df["eui_quartile"] = pd.qcut(completed[kpi_col], q=4, labels=["Q1", "Q2", "Q3", "Q4"])

fig, ax = plt.subplots(figsize=(14, 7))
parallel_coordinates(
    norm_df,
    "eui_quartile",
    ax=ax,
    alpha=0.3,
    colormap="RdYlGn_r",
)
ax.set_ylabel("Normalised Value", fontsize=12)
ax.set_title("Parallel Coordinates: Parameters + EUI (coloured by EUI quartile)", fontsize=13)
ax.legend(title="EUI Quartile", fontsize=10)
fig.tight_layout()
plt.show()

In [ ]:
# --- Box Plot by HVAC Type (replaces PAT: Results > Box Plot) ---
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=completed, x="hvac_type", y="eui_kwh_m2_yr", palette="Set2", ax=ax)
sns.stripplot(
    data=completed, x="hvac_type", y="eui_kwh_m2_yr", color="black", alpha=0.3, size=3, ax=ax
)
ax.set_xlabel("HVAC System Type", fontsize=12)
ax.set_ylabel("EUI (kWh/m²/yr)", fontsize=12)
ax.set_title("EUI Distribution by HVAC System Type", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# --- Correlation Heatmap (replaces PAT: Results > Correlation Matrix) ---
numeric_cols = completed.select_dtypes(include=[np.number]).columns
corr = completed[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Parameter Correlation Matrix", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# --- Parameter Sensitivity Bar Chart (openstudio-server didn't have this) ---
param_cols_for_sensitivity = ["insul_r", "wwr_south", "cooling_setpoint", "lighting_power"]
correlations = {
    col: abs(completed[col].corr(completed["eui_kwh_m2_yr"])) for col in param_cols_for_sensitivity
}
cor_series = pd.Series(correlations).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
cor_series.plot.barh(ax=ax, color="steelblue", edgecolor="white")
ax.set_xlabel("Absolute Pearson Correlation with EUI", fontsize=12)
ax.set_title("Parameter Sensitivity Ranking", fontsize=14)
ax.axvline(0.3, color="gray", linestyle="--", alpha=0.5, label="Weak threshold")
ax.axvline(0.5, color="orange", linestyle="--", alpha=0.5, label="Moderate threshold")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# --- Pair Plot (replaces PAT: Results > Scatter Matrix) ---
scatter_cols = ["insul_r", "wwr_south", "cooling_setpoint", "eui_kwh_m2_yr"]
g = sns.pairplot(
    completed[scatter_cols],
    diag_kind="kde",
    corner=True,
    plot_kws={"alpha": 0.4, "s": 15},
)
g.figure.suptitle("Parameter Pair Plot", y=1.02, fontsize=14)
plt.show()

---
## 5. Boilerplate Snippets

Copy these cells into your own notebooks. Each is self-contained.

In [ ]:
# BOILERPLATE: Quick load with DuckDB
import duckdb

con = duckdb.connect()
df = con.execute("""
    SELECT * FROM read_parquet('results/aggregated_results.parquet')
    WHERE status = 'completed'
""").fetchdf()
print(f"{len(df)} completed samples loaded")

In [ ]:
# BOILERPLATE: Top/bottom N samples by KPI
KPI = "eui_kwh_m2_yr"
N = 10

print(f"--- Top {N} by {KPI} ---")
print(df.nlargest(N, KPI)[["sample_id", KPI]])
print(f"\n--- Bottom {N} by {KPI} ---")
print(df.nsmallest(N, KPI)[["sample_id", KPI]])

In [ ]:
# BOILERPLATE: Export filtered subset to CSV
subset = df[df["eui_kwh_m2_yr"] < 150]
subset.to_csv("results/low_eui_subset.csv", index=False)
print(f"Exported {len(subset)} samples with EUI < 150")

In [ ]:
# BOILERPLATE: Run.json summary (replaces dashboard status)
import json

run_json_path = Path("results/run.json")
if run_json_path.exists():
    with open(run_json_path) as f:
        run_data = json.load(f)

    summary = run_data.get("summary", {})
    print("Campaign Summary:")
    print(f"  Total samples: {summary.get('n_samples', 'N/A')}")
    print(f"  Succeeded: {summary.get('n_succeeded', 'N/A')}")
    print(f"  Failed: {summary.get('n_failed', 'N/A')}")
    print(f"  Wall clock: {summary.get('wall_clock_s', 'N/A')}s")

    # Per-step timing
    steps = run_data.get("steps", [])
    if steps:
        print("\nPer-step timing:")
        for step in steps:
            name = step.get("name", "unknown")
            elapsed = step.get("elapsed_s", 0)
            print(f"  {name}: {elapsed:.2f}s")
else:
    print("run.json not found — campaign may not have completed yet")

In [ ]:
# BOILERPLATE: Compare two campaigns (replaces MongoDB cross-analysis)
df_v1 = pd.read_csv("results_v1/aggregated_results.csv")
df_v2 = pd.read_csv("results_v2/aggregated_results.csv")

print("Campaign Comparison:")
print(f"  V1: mean EUI = {df_v1['eui_kwh_m2_yr'].mean():.1f}, n = {len(df_v1)}")
print(f"  V2: mean EUI = {df_v2['eui_kwh_m2_yr'].mean():.1f}, n = {len(df_v2)}")
print(
    f"  Improvement: {df_v1['eui_kwh_m2_yr'].mean() - df_v2['eui_kwh_m2_yr'].mean():.1f} kWh/m²/yr"
)

---
## References

- [Migration Guide](../../docs/migration-openstudio-server.md) — Full openstudio-server to OSimFlow migration
- [R to Python BYOS Cheatsheet](../user_scripts/examples/r_to_python_migration.md) — Worked R→Python examples
- [eplusout.sql Guide](../../docs/eplusout-sql-guide.md) — Querying EnergyPlus SQL output
- [DuckDB Documentation](https://duckdb.org/docs/) — SQL on Parquet files
- [pandas Documentation](https://pandas.pydata.org/docs/) — Data analysis library